In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STANDALONE BITCOIN MANUAL-BUBBLE MODEL
# Self-contained: loads data, fits power-law support, then fits bubble_shape()
# to each bubble year the user specifies in BUBBLE_YEARS.  No rolling-slope
# phase classification is performed — you choose the peak years; the model
# determines the rise, plateau, and decay parameters automatically.
#
# Workflow:
#   1. Fit power-law support line to historical data.
#   2. For each year in BUBBLE_YEARS, locate the local log-excess peak inside a
#      ±BUBBLE_YEAR_WINDOW search window and fit bubble_shape() sequentially
#      to the residual log-excess (largest peak first).
#   3. Classify fitted bubbles as MAJOR (top N_MAJOR by peak K) / MINOR.
#   4. Compute composite model R².
#   5. Extrapolate or average future bubble parameters and plot projections.
#
# No other cells need to have run first.
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution
from scipy.stats import linregress
from matplotlib.ticker import FixedLocator, StrMethodFormatter, NullFormatter
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

# ── Data & input ──────────────────────────────────────────────────────────────
csv_path       = './BitcoinPricesDaily.csv'
FIT_MIN_DATE   = '2010-01-01'   # exclude early near-zero trades from the fit
FIT_MAX_DATE   = None           # None = use all available data
MIN_DATA_YEARS = 1.0
# Bitcoin had no real market price in its first year (genesis 2009-01-09 → ~2010).
# Excluding it prevents the power-law support fit from being skewed by near-zero trades.

# ── Support line ──────────────────────────────────────────────────────────────
SUPPORT_PERCENTILE = 25   # bottom X% of OLS residuals → support candidates
SUPPORT_QUANTILE   = 0.50

# ── Bubble years ──────────────────────────────────────────────────────────────
BUBBLE_YEARS = [2013, 2017, 2021, 2025]
# List the calendar years in which you believe a Bitcoin bubble peaked.
# For each year, the model searches for the highest log-excess (price/support)
# within ±BUBBLE_YEAR_WINDOW of Jan 1 of that year.
#
# Tips:
#   • Include separate entries for close double-peaks (e.g. 2013 and 2014 for
#     the April-2013 and December-2013 peaks).
#   • Years with no historical data in their window are automatically skipped
#     (no error).  Predicted bubbles are added via STEP 6 extrapolation.
#   • Order does not matter — the model sorts by peak magnitude before fitting.

BUBBLE_YEAR_WINDOW = 0.5
# Half-width of the search window, in years.
# A value of 1.0 means the model searches within ±1 year of Jan 1 of each
# bubble year, i.e. a 2-year window centred on Jan 1 of that year.
#
# Larger values help when the actual peak falls far from Jan 1 (e.g. a
# December peak in a "year" bubble).  Smaller values prevent adjacent
# bubble years from sharing data points.  Windows may overlap — the
# sequential residual fitting handles this correctly.

# ── Fitting ───────────────────────────────────────────────────────────────────
#FIT_CONTEXT_YR      = 1.0 #default=1.0
FIT_CONTEXT_YR      = 0.5
# Extra data (in years) included beyond the ±BUBBLE_YEAR_WINDOW on each side
# when optimising a bubble's parameters.  This context shows the optimiser
# where the residual returns to zero, anchoring the rise and decay tails.
# Increase if bubble fits look truncated; decrease if adjacent bubbles bleed in.

FIT_RISE_LOOKBACK_YR = 0.5
# The bubble's rise can start this many years before the left edge of the
# search window.  Needed because the actual price rise begins before the
# peak window — the bubble was already building up before the window opens.

PLATEAU_PARALLEL_SUPPORT = False
# True  → plat_pow is fixed at 0 (plateau price grows at exactly the support
#          power-law rate; log-excess is flat during the plateau).  Fewer
#          parameters, more stable fits, consistent with the phase model.
# False → plat_pow is a free parameter (6-D optimisation).  Allows the
#          plateau to tilt: < 0 = rounded top, > 0 = blow-off acceleration.

PLAT_POW_RANGE = 8.0
# |plat_pow| upper bound during optimisation and prediction clipping.
# ±slope_sup ≈ ±5.8 spans "price constant" to "double support slope"; 8.0
# gives modest headroom beyond the physically meaningful range.

DE_MAXITER = 2000
# Maximum iterations for the Differential Evolution optimiser per bubble.
# With 5–6 free parameters, 2000 iterations is a good balance of speed and
# solution quality.  Increase to 4000+ if fit costs look suspiciously high.

DE_POPSIZE = 18
# Population multiplier for DE.  Population size = DE_POPSIZE × n_params.
# With 5–6 params, this gives 90–108 candidates per generation.
# DE recommends at least 10–15×; 18 is a safe conservative choice.

# ── Classification ────────────────────────────────────────────────────────────
N_MAJOR = 5
# Top N_MAJOR fitted bubbles by peak K are labelled "major"; the rest "minor".
# Bitcoin's canonical halving-driven cycles: 2011, 2013, 2017, 2021.
# Set to 5 to also include 2025 as a major bubble.

MAX_MAJOR_BUBBLES = None
# After classification, keep only the N highest-K major bubbles in the model.
# None = keep all N_MAJOR.  Example: set to 3 to drop the weakest major.

MAX_MINOR_BUBBLES = 8
# Keep only the N highest-K minor bubbles in the model.
# None = keep all.  Reduces clutter in plots and speeds up prediction.

# ── Prediction ────────────────────────────────────────────────────────────────
N_PREDICT_MAJOR      = 2     # number of future major bubbles to project
N_PREDICT_MINOR      = 4     # number of future minor bubbles to project

PREDICT_MODE         = 'extrap'
# 'avg'    — each predicted bubble gets the weighted-average parameters of the
#            last PREDICT_LAST_N_MAJOR historical bubbles.  All predictions are
#            identical in shape; only the start time shifts.  Stable / conservative.
# 'extrap' — parameters are linearly extrapolated along their historical trend
#            (r and d log-linearly; K, dur_plateau, plat_pow linearly).

PREDICT_LAST_N_MAJOR = 3
# Only the most recent N major bubbles feed the avg / extrapolation.
# None = use all detected major bubbles.
# Setting this to 3-4 focuses on recent cycles and ignores the very different
# early history (e.g. the 2011 bubble with r≈4).

PREDICT_LAST_N_MINOR = 4   # same idea for minor bubbles

MAJOR_INTERVAL_YR        = 3.8
# Fallback inter-bubble interval (years) used when fewer than 2 historical
# intervals can be measured (i.e. only 1 historical major bubble exists).

MAJOR_INTERVAL_USE_TREND = True
# False → predicted interval = weighted average of historical intervals (stable).
# True  → interval is also linearly extrapolated along its historical trend,
#         so predicted cycles can lengthen or shorten over time.

MAJOR_EXTRAP_WEIGHTS     = None
# Per-bubble, per-parameter importance weights for the avg / extrapolation.
# None = uniform (every bubble and every parameter contributes equally).
#
# If set, must be a flat list of length N_major × 6, where N_major is the
# number of historical major bubbles actually used (after PREDICT_LAST_N_MAJOR
# trimming).  The layout is row-major: one row of 6 weights per bubble,
# ordered chronologically (oldest first):
#
#   [w_r, w_d, w_K, w_interval, w_dur_plateau, w_plat_pow]   ← bubble 1 (oldest)
#   [w_r, w_d, w_K, w_interval, w_dur_plateau, w_plat_pow]   ← bubble 2
#   ...
#
# Parameter roles:
#
#   r  (rise rate, yr⁻¹)
#       Controls how steeply price climbs above support during the rise phase.
#       High r → sharp, spike-like rally.  r has historically declined each
#       cycle (early bubbles were explosive; recent ones are more gradual).
#       Extrapolated log-linearly so it can only approach zero, never go negative.
#
#   d  (decay rate, yr⁻¹)
#       Controls how quickly price falls back toward support after the peak.
#       High d → short, sharp crash.  Like r, d has trended downward over time
#       (bear markets are getting longer).  Also log-linear.
#
#   K  (peak log-excess, log₁₀ units)
#       log₁₀(peak_price / support_price_at_peak).  K=1 means the peak was
#       10× the support line; K=0.5 means ~3×.  K has declined each cycle
#       as Bitcoin matures — predicting its future value is the key uncertainty.
#       Extrapolated log-linearly (geometric mean in avg mode): each cycle's K
#       is a roughly constant fraction of the prior one, so log(K) is linear
#       in cycle index.  This also ensures predicted K is always positive.
#       A higher weight on recent bubbles will anchor K to the current regime.
#
#   interval  (time between successive t_rise values, years)
#       How many years elapse between the start of one bubble and the next.
#       Used to position the predicted bubble in time.  The weight column here
#       applies to the *intervals* array (one shorter than the bubble array),
#       so the last bubble's w_interval weight is effectively unused.
#
#   dur_plateau  (plateau duration, years)
#       How long price stays near the peak before the decay begins.  Short
#       plateaus give sharp V-shaped peaks; longer ones give extended tops.
#       Has been roughly stable across cycles, so averaging is usually fine.
#
#   plat_pow  (plateau power-law tilt, dimensionless)
#       Differential exponent during the plateau relative to the support slope.
#       plat_pow=0 → price grows at exactly the support rate (flat log-excess).
#       plat_pow<0 → price drifts down during the plateau (rounded top).
#       plat_pow>0 → price accelerates beyond support growth (blow-off top).
#       Currently fixed at 0 when PLATEAU_PARALLEL_SUPPORT=True.
#
# Example — downweight the oldest bubble and ignore plat_pow entirely:
#   MAJOR_EXTRAP_WEIGHTS = [0.5, 0.5, 0.5, 0.5, 0.5, 0,   # bubble 1 (old)
#                           1.0, 1.0, 1.0, 1.0, 1.0, 0,   # bubble 2
#                           1.5, 1.5, 1.5, 1.5, 1.5, 0]   # bubble 3 (recent)

MINOR_EXTRAP_WEIGHTS     = None  # same layout as MAJOR_EXTRAP_WEIGHTS
MINOR_INTERVAL_USE_TREND = True

# ── Bubble model colours ──────────────────────────────────────────────────────
DATA_COLOR   = '#1D4ED8'   # historical daily price scatter
MAJOR_COLORS = ['#EF4444', '#F97316', '#EAB308', '#22C55E',
                '#3B82F6', '#8B5CF6', '#EC4899', '#14B8A6']
MINOR_COLORS = ['#FCA5A5', '#FDBA74', '#FDE68A', '#86EFAC',
                '#93C5FD', '#C4B5FD', '#F9A8D4', '#99F6E4']

# ── Shared chart colours ──────────────────────────────────────────────────────
SUPPORT_COLOR        = '#3B82F6'   # support power-law line
COMPOSITE_COLOR      = '#DC2626'   # composite model curve
TODAY_COLOR          = '#888888'   # "today" vertical marker
FIT_MIN_COLOR        = '#10B981'   # FIT_MIN_DATE vertical marker
FIT_MAX_COLOR        = '#EF4444'   # FIT_MAX_DATE vertical marker
SCATTER_COLOR        = '#94A3B8'   # raw data scatter in residual / excess plots
EXCESS_SMOOTH_COLOR  = '#2563EB'   # smoothed excess line in decomposition plot
RESID_SUP_COLOR      = '#3B82F6'   # smoothed support-only residual line
RESID_COMP_COLOR     = '#EF4444'   # smoothed composite residual line
REFERENCE_LINE_COLOR = 'black'     # zero-reference horizontal lines
GRID_MAJOR_COLOR     = '#CCCCCC'   # major grid lines
GRID_MINOR_COLOR     = '#E5E5E5'   # minor grid lines
PLOT_BG_COLOR        = 'white'     # axes background colour
FALLBACK_TEXT_COLOR  = 'gray'      # "no data" placeholder text

# ── Plot sizing ───────────────────────────────────────────────────────────────
FIGSIZE_WIDE      = (15, 9)    # log-log, semi-log, and most chart panels
FIGSIZE_DECOMP    = (15, 10)   # bubble decomposition (2 stacked panels)
FIGSIZE_RESIDUALS = (15, 8)    # residual comparison (2 panels)
FIGSIZE_ENVELOPE  = (13, 7)    # peak-aligned envelope chart

# ── Plot axes ─────────────────────────────────────────────────────────────────
PLOT_YEARS_MIN         = 1.0    # first year on the dense plot grid and full x-axis
PLOT_YEARS_MAX         = 42.0   # last year on the dense plot grid and full x-axis
PLOT_GRID_POINTS       = 3000   # number of points in the dense time-grid
PRICE_YMIN             = 0.01   # lower price y-axis limit (USD)
PRICE_YMAX             = 1e8    # upper price y-axis limit (USD)
ZOOM_XMIN              = 15     # zoomed chart left bound (years since genesis)
ZOOM_XMAX              = 35     # zoomed chart right bound (years since genesis)
ZOOM_YMIN              = 2e4    # zoomed chart lower price limit (USD)
ZOOM_YMAX              = 3e7    # zoomed chart upper price limit (USD)
RESIDUAL_SMOOTH_WINDOW = 90     # rolling-mean window (days) for residual plots


# ══════════════════════════════════════════════════════════════════════════════
# LOAD + PREPARE DATA
# ══════════════════════════════════════════════════════════════════════════════
try:
    df = pd.read_csv(csv_path)
    print(f"CSV loaded.  Shape: {df.shape}")
except FileNotFoundError:
    print(f"File not found: {csv_path}")

df.columns = ['Date', 'Price']
df['Date']  = pd.to_datetime(df['Date'], format='%m/%d/%y', errors='coerce')
df = df.dropna(subset=['Date']).sort_values('Date')

x_dates = df['Date']
y_data  = df['Price'].astype(float)

genesis     = pd.to_datetime('2009-01-09')
years_since = (x_dates - genesis).dt.days / 365.25

valid       = y_data > 0
years_valid = years_since[valid].astype(float)
y_valid     = y_data[valid]

reasonable = years_valid >= MIN_DATA_YEARS
years_all  = years_valid[reasonable].values
y_all      = y_valid[reasonable].values
dates_all  = x_dates[valid][reasonable].values

log_t_all = np.log10(years_all)
log_p_all = np.log10(y_all)

fit_mask = np.ones(len(dates_all), dtype=bool)
if FIT_MIN_DATE:
    fit_mask &= pd.to_datetime(dates_all) >= pd.to_datetime(FIT_MIN_DATE)
if FIT_MAX_DATE:
    fit_mask &= pd.to_datetime(dates_all) <= pd.to_datetime(FIT_MAX_DATE)

log_t     = log_t_all[fit_mask]
log_p     = log_p_all[fit_mask]
years_fit = years_all[fit_mask]
dates_fit = dates_all[fit_mask]

today_years   = (pd.to_datetime('today') - genesis).days / 365.25
fit_min_years = (pd.to_datetime(FIT_MIN_DATE) - genesis).days / 365.25 if FIT_MIN_DATE else None
fit_max_years = (pd.to_datetime(FIT_MAX_DATE) - genesis).days / 365.25 if FIT_MAX_DATE else None

print(f"Date range  : {pd.Timestamp(dates_all[0]).date()} → {pd.Timestamp(dates_all[-1]).date()}")
print(f"Fit window  : {fit_mask.sum()} points")
print(f"Price range : ${y_all.min():,.2f} – ${y_all.max():,.2f}")
print(f"Today       : t = {today_years:.3f} yr")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 1: FIT SUPPORT LINE
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print(f"STEP 1: FITTING SUPPORT LINE (bottom {SUPPORT_PERCENTILE}% OLS residual)")
print("=" * 80)

slope_ols, intercept_ols, _, _, _ = linregress(log_t, log_p)
ols_resid    = log_p - (intercept_ols + slope_ols * log_t)
cutoff       = np.percentile(ols_resid, SUPPORT_PERCENTILE)
support_mask = ols_resid <= cutoff

X_support = sm.add_constant(log_t[support_mask])
res_sup   = sm.QuantReg(log_p[support_mask], X_support).fit(q=SUPPORT_QUANTILE, max_iter=10000)

intercept_sup = res_sup.params[0]
slope_sup     = res_sup.params[1]
A_sup         = 10 ** intercept_sup
B_sup         = slope_sup

print(f"  Support points : {support_mask.sum()} / {len(log_t)}")
print(f"  Power law      : price = {A_sup:.4e} × t^{B_sup:.4f}")

log_support_all = intercept_sup + slope_sup * log_t_all
log_support_fit = intercept_sup + slope_sup * log_t

log_excess_all = log_p_all - log_support_all
log_excess_fit = log_p     - log_support_fit


# ══════════════════════════════════════════════════════════════════════════════
# BUBBLE SHAPE FUNCTION  (used by STEP 3 below)
# ══════════════════════════════════════════════════════════════════════════════
def bubble_shape(t, t_rise, r, t_plateau, t_decay, d, plat_pow=0.0):
    """
    Log₁₀ bubble excess above the power-law support line.

    Parameters
    ----------
    t         : array of time values (years since genesis)
    t_rise    : start of the exponential rise
    r         : rise rate (yr⁻¹); price doubles relative to support every log(2)/r years
    t_plateau : end of rise / start of plateau
    t_decay   : end of plateau / start of decay
    d         : decay rate (yr⁻¹)
    plat_pow  : differential power-law exponent during plateau, relative to support.
                price ∝ t^(slope_sup + plat_pow) during [t_plateau, t_decay].
                  plat_pow = 0          → plateau parallels support (log_excess = K)
                  plat_pow = −slope_sup → price is constant during plateau
                  plat_pow > 0          → blow-off top (price grows faster than support)

    Phases
    ------
    Rise    [t_rise, t_plateau):
        log_excess(t) = r·(t − t_rise) + slope_sup·log₁₀(t_rise / t)
        Equivalently: price grows exponentially from support(t_rise).

    Plateau [t_plateau, t_decay):
        log_excess(t) = K + plat_pow·log₁₀(t / t_plateau)
        where K = log_excess(t_plateau) = peak of rise phase.
        plat_pow = 0 recovers the original "plateau parallels support" behaviour.

    Decay   [t_decay, ∞):
        log_excess(t) = K_end − d·(t − t_decay) + slope_sup·log₁₀(t_decay / t), clipped ≥ 0
        where K_end = K + plat_pow·log₁₀(t_decay / t_plateau) = log_excess at t_decay.
        Price decays exponentially back toward support(t_decay).
    """
    t = np.asarray(t, dtype=float)
    result = np.zeros_like(t)
    if t_plateau <= t_rise:
        return result
    # K: log-excess at the start of the plateau (= peak of rise)
    K = r * (t_plateau - t_rise) + slope_sup * np.log10(
        np.maximum(t_rise / t_plateau, 1e-12))
    if K <= 0:
        return result
    # K_end: log-excess at the end of the plateau (= start of decay)
    K_end = (K + plat_pow * np.log10(np.maximum(t_decay / t_plateau, 1e-12))
             if t_decay > t_plateau else K)

    # Rise
    m = (t >= t_rise) & (t < t_plateau)
    if m.any():
        result[m] = np.maximum(
            r * (t[m] - t_rise) + slope_sup * np.log10(
                np.maximum(t_rise / t[m], 1e-12)), 0.0)
    # Plateau (general power law)
    m = (t >= t_plateau) & (t < t_decay)
    if m.any():
        result[m] = np.maximum(
            K + plat_pow * np.log10(np.maximum(t[m] / t_plateau, 1e-12)), 0.0)
    # Decay
    m = t >= t_decay
    if m.any():
        result[m] = np.maximum(
            K_end - d * (t[m] - t_decay) + slope_sup * np.log10(
                np.maximum(t_decay / t[m], 1e-12)), 0.0)
    return result


# ══════════════════════════════════════════════════════════════════════════════
# STEP 2: LOCATE BUBBLE PEAKS FROM BUBBLE_YEARS
# ══════════════════════════════════════════════════════════════════════════════
# For each year in BUBBLE_YEARS, find the highest log-excess (price/support)
# within a ±BUBBLE_YEAR_WINDOW window centred on Jan 1 of that year.
#
# The search window is just for locating the peak — the fitting window in
# STEP 3 is wider (extends by ±FIT_CONTEXT_YR beyond the search window) so
# the optimiser sees the residual approaching zero on both sides.
#
# Years where the search window falls entirely outside the available data are
# silently skipped.  They will not produce a fitted bubble — use the STEP 6
# prediction to project those future peaks.
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print(f"STEP 2: LOCATE BUBBLE PEAKS  (BUBBLE_YEAR_WINDOW = ±{BUBBLE_YEAR_WINDOW} yr)")
print("=" * 80)
print(f"\n  Searching for peaks near: {BUBBLE_YEARS}")
print(f"\n  {'Year':>4}  {'t_center':>8}  {'Search window':>22}  {'Peak date':>12}  "
      f"{'t_peak':>7}  {'Raw K':>6}")
print("  " + "-" * 72)

bm_peaks = []   # list of dicts describing each located peak

for yr in BUBBLE_YEARS:
    # Jan 1 of the specified year, converted to years since genesis
    t_center = (pd.to_datetime(f'{yr}-01-01') - genesis).days / 365.25
    t_lo     = t_center - BUBBLE_YEAR_WINDOW
    t_hi     = t_center + BUBBLE_YEAR_WINDOW

    # Only search within available data
    window_mask = (years_fit >= t_lo) & (years_fit <= t_hi)
    if not window_mask.any():
        print(f"  {yr:>4}  {t_center:>8.3f}  [{t_lo:>8.3f}, {t_hi:>8.3f}]  "
              f"{'(no data — skipped)':>42}")
        continue

    local_exc = log_excess_fit[window_mask]
    local_yrs = years_fit[window_mask]
    local_dt  = dates_fit[window_mask]
    peak_i    = int(np.argmax(local_exc))
    peak_t    = local_yrs[peak_i]
    peak_K    = local_exc[peak_i]
    peak_date = pd.Timestamp(local_dt[peak_i]).strftime('%Y-%m-%d')

    bm_peaks.append({
        'bubble_year': yr,
        'peak_t':      peak_t,     # years since genesis at the log-excess peak
        'region_lo':   t_lo,       # left edge of search window
        'region_hi':   t_hi,       # right edge of search window
        'raw_K':       peak_K,
    })
    print(f"  {yr:>4}  {t_center:>8.3f}  [{t_lo:>8.3f}, {t_hi:>8.3f}]  "
          f"{peak_date:>12}  {peak_t:>7.3f}  {peak_K:>6.3f}")

if not bm_peaks:
    print("\n  No peaks found in any window.  Check BUBBLE_YEARS and BUBBLE_YEAR_WINDOW.")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 3: FIT BUBBLE_SHAPE() TO EACH LOCATED PEAK
# ══════════════════════════════════════════════════════════════════════════════
# Fitting strategy: sequential residual fitting, largest raw peak first.
#
# Each bubble is fitted to the CURRENT residual (raw log-excess minus the
# shapes already fitted).  After fitting, this bubble's shape is subtracted
# from the residual before the next bubble is fitted.  This prevents
# double-counting: without it, all bubbles independently fit the full signal
# and their sum far exceeds the data.
#
# The optimisation window extends FIT_CONTEXT_YR beyond the search window so
# the optimiser sees the residual returning to zero on both sides.  The rise
# start (t_rise) can begin up to FIT_RISE_LOOKBACK_YR before the left edge
# of the search window, because the actual rally precedes the peak window.
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("STEP 3: FITTING BUBBLE SHAPES")
print("=" * 80)
print(f"  plat_pow: {'fixed = 0 (plateau parallels support)' if PLATEAU_PARALLEL_SUPPORT else 'free parameter'}")
print(f"  FIT_CONTEXT_YR = {FIT_CONTEXT_YR}  FIT_RISE_LOOKBACK_YR = {FIT_RISE_LOOKBACK_YR}")
print(f"  DE_MAXITER = {DE_MAXITER}  DE_POPSIZE = {DE_POPSIZE}")


def fit_manual_bubble(pk, idx, residual):
    """
    Fit bubble_shape() to one manually-located peak.

    Parameters
    ----------
    pk       : peak dict from bm_peaks (keys: peak_t, region_lo, region_hi, raw_K)
    idx      : integer index for RNG seeding (ensures reproducibility)
    residual : current residual log-excess (raw log-excess minus previously
               fitted bubble contributions)

    Returns
    -------
    dict of bubble parameters, dates, and diagnostics
    """
    region_lo = pk['region_lo']
    region_hi = pk['region_hi']
    peak_t    = pk['peak_t']

    # ── Fitting data window ──────────────────────────────────────────────────
    # Extend beyond the search window so the optimiser sees the return to zero.
    t_lo = max(years_fit[0],  region_lo - FIT_CONTEXT_YR)
    t_hi = min(years_fit[-1], region_hi + FIT_CONTEXT_YR)
    ctx  = (years_fit >= t_lo) & (years_fit <= t_hi)
    t_ctx = years_fit[ctx]
    exc   = np.maximum(0.0, residual[ctx])   # fit to non-negative residual

    span = (region_hi - region_lo) + 2 * FIT_CONTEXT_YR   # total fitting span

    # ── Optimisation bounds ──────────────────────────────────────────────────
    # t_rise: start from FIT_RISE_LOOKBACK_YR before window, up to the peak
    t_rise_lb = max(years_fit[0], region_lo - FIT_RISE_LOOKBACK_YR)

    bounds_5 = [
        (t_rise_lb, peak_t),    # t_rise  — rise begins before or at the peak
        (0.05, 20.0),            # r       — rise rate (yr⁻¹)
        (0.02, span),            # dur_rise — rise duration
        (0.0,  span),            # dur_plateau — 0 = no plateau (V-shaped peak)
        (0.05, 20.0),            # d       — decay rate (yr⁻¹)
    ]

    if PLATEAU_PARALLEL_SUPPORT:
        # 5-D optimisation: plat_pow is fixed at 0
        def obj5(p5):
            tr, r_, dr, dp, d_ = p5
            pred = bubble_shape(t_ctx, tr, r_, tr + dr, tr + dr + dp, d_, 0.0)
            return float(np.sum((exc - pred) ** 2))

        res    = differential_evolution(obj5, bounds_5, maxiter=DE_MAXITER,
                                        popsize=DE_POPSIZE, seed=42 + idx,
                                        tol=1e-10, polish=True)
        params = list(res.x) + [0.0]   # append plat_pow = 0
        cost   = res.fun

    else:
        # 6-D optimisation: plat_pow is a free parameter
        bounds_6 = bounds_5 + [(-PLAT_POW_RANGE, PLAT_POW_RANGE)]

        def obj6(p6):
            tr, r_, dr, dp, d_, pp = p6
            pred = bubble_shape(t_ctx, tr, r_, tr + dr, tr + dr + dp, d_, pp)
            return float(np.sum((exc - pred) ** 2))

        res    = differential_evolution(obj6, bounds_6, maxiter=DE_MAXITER,
                                        popsize=DE_POPSIZE, seed=42 + idx,
                                        tol=1e-10, polish=True)
        params = list(res.x)
        cost   = res.fun

    # ── Unpack and derive K values ───────────────────────────────────────────
    tr, r_, dr, dp, d_, pp = params
    tplat  = tr + dr
    tdec   = tplat + dp
    # K_peak: log-excess at the end of the rise (= start of plateau)
    K_peak = max(r_ * dr + slope_sup * np.log10(max(tr / tplat, 1e-12)), 0.0)
    # K_end: log-excess at the end of the plateau (= start of decay)
    K_end  = K_peak + pp * np.log10(max(tdec / tplat, 1e-12)) if dp > 0 else K_peak
    # K_bubble: the maximum log-excess (peak of the whole bubble)
    K_bub  = max(K_peak, K_end)
    # t_end: approximate time when the decay returns to support (log_excess → 0)
    t_end  = tdec + (max(K_end, 0.0) / d_ if d_ > 0 else 0.0)

    return {
        't_rise':      tr,      'r':           r_,    'dur_rise':    dr,
        't_plateau':   tplat,   'dur_plateau':  dp,   't_decay':     tdec,
        'd':           d_,      'plat_pow':     pp,
        'K':           K_bub,   'K_peak':       K_peak, 'K_end':     K_end,
        't_end':       t_end,   'cost':         cost,
        't_start':     tr,      # alias used by plot helpers
        'bubble_year': pk['bubble_year'],
        'date_rise':   genesis + pd.Timedelta(days=tr    * 365.25),
        'date_plat':   genesis + pd.Timedelta(days=tplat * 365.25),
        'date_decay':  genesis + pd.Timedelta(days=tdec  * 365.25),
        'date_end':    genesis + pd.Timedelta(days=t_end  * 365.25),
    }


# ── Sequential residual fitting: largest raw peak first ──────────────────────
# Sorting by raw_K descending ensures dominant events are captured cleanly
# before smaller ones.  This mirrors how the phase model fits bubbles.
peaks_by_magnitude = sorted(
    enumerate(bm_peaks),
    key=lambda x: x[1]['raw_K'],
    reverse=True
)

residual_bm  = log_excess_fit.copy()   # mutable residual; starts as raw log-excess
bm_fitted    = []                      # fitted bubble parameter dicts

for rank, (orig_idx, pk) in enumerate(peaks_by_magnitude):
    print(f"  Fitting bubble year {pk['bubble_year']} "
          f"({rank + 1}/{len(bm_peaks)}, raw K={pk['raw_K']:.3f}) ...",
          end=' ', flush=True)
    bp = fit_manual_bubble(pk, orig_idx, residual_bm)
    bm_fitted.append(bp)
    # Subtract this bubble's contribution from the residual
    contrib     = bubble_shape(years_fit, bp['t_rise'], bp['r'],
                               bp['t_plateau'], bp['t_decay'], bp['d'],
                               bp.get('plat_pow', 0.0))
    residual_bm = np.maximum(0.0, residual_bm - contrib)
    pp_str = (f"plat_pow={bp['plat_pow']:+.2f}"
              if not PLATEAU_PARALLEL_SUPPORT else "plat_pow=0")
    print(f"K={bp['K']:.3f} ({10**bp['K']:.1f}×)  {pp_str}  "
          f"cost={bp['cost']:.4f}  resid_max={residual_bm.max():.3f}")

# Sort chronologically by rise time for display and classification
bm_fitted.sort(key=lambda b: b['t_rise'])


# ══════════════════════════════════════════════════════════════════════════════
# STEP 4: CLASSIFY FITTED BUBBLES → MAJOR / MINOR
# ══════════════════════════════════════════════════════════════════════════════
# The top N_MAJOR bubbles by peak K are labelled "major"; the rest are "minor".
# Optional caps (MAX_MAJOR_BUBBLES, MAX_MINOR_BUBBLES) further trim each class
# to the strongest K values so plots stay readable.
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("STEP 4: MAJOR / MINOR CLASSIFICATION")
print("=" * 80)

n_det   = len(bm_fitted)
n_maj   = min(N_MAJOR, n_det)
by_K    = sorted(range(n_det), key=lambda i: bm_fitted[i]['K'], reverse=True)
maj_set = set(by_K[:n_maj])

bm_major = sorted([bm_fitted[i] for i in maj_set],
                   key=lambda b: b['t_rise'])
bm_minor = sorted([bm_fitted[i] for i in range(n_det) if i not in maj_set],
                   key=lambda b: b['t_rise'])

# Apply optional caps (keep highest-K within each class, re-sort chronologically)
if MAX_MAJOR_BUBBLES is not None and len(bm_major) > MAX_MAJOR_BUBBLES:
    bm_major = sorted(sorted(bm_major, key=lambda b: -b['K'])[:MAX_MAJOR_BUBBLES],
                      key=lambda b: b['t_rise'])
if MAX_MINOR_BUBBLES is not None and len(bm_minor) > MAX_MINOR_BUBBLES:
    bm_minor = sorted(sorted(bm_minor, key=lambda b: -b['K'])[:MAX_MINOR_BUBBLES],
                      key=lambda b: b['t_rise'])

print(f"  {n_det} total  →  {len(bm_major)} MAJOR + {len(bm_minor)} MINOR")
print(f"  (top {N_MAJOR} by peak K = major"
      + (f"; capped at {MAX_MAJOR_BUBBLES}" if MAX_MAJOR_BUBBLES else "")
      + f";  {len(bm_minor)} minor shown"
      + (f" of {n_det - len(maj_set)}"
         if MAX_MINOR_BUBBLES and n_det - len(maj_set) > len(bm_minor) else "")
      + ")")


def print_bm_table(params, label):
    """
    Print a formatted table of fitted bubble parameters.

    Columns: bubble year, rise date, rise rate r (yr⁻¹), rise duration (yr),
    peak K (log₁₀ multiplier above support), price peak as a multiple of support,
    plateau duration (yr), decay rate d (yr⁻¹), optionally plat_pow, and the
    inter-bubble interval Δt_rise (yr) from the previous bubble.
    """
    if not params:
        print(f"\n  {label}: none"); return
    print(f"\n  {label} BUBBLES:")
    hdr = (f"  {'Yr':>4}  {'date_rise':>12}  {'r':>6}  {'dur_rise':>8}  "
           f"{'K':>6}  {'Peak×':>6}  {'dur_plat':>8}  {'d':>6}")
    if not PLATEAU_PARALLEL_SUPPORT:
        hdr += f"  {'plat_pow':>8}"
    hdr += f"  {'Δt_rise':>8}"
    print(hdr)
    print("  " + "-" * (len(hdr) - 2))
    for i, bp in enumerate(params):
        dt  = f"{bp['t_rise'] - params[i-1]['t_rise']:.2f} yr" if i > 0 else ""
        row = (f"  {bp.get('bubble_year','?'):>4}  "
               f"{bp['date_rise'].strftime('%Y-%m-%d'):>12}  "
               f"{bp['r']:>6.3f}  {bp['dur_rise']:>8.3f}  {bp['K']:>6.3f}  "
               f"{10**bp['K']:>6.1f}×  {bp['dur_plateau']:>8.3f}  {bp['d']:>6.3f}")
        if not PLATEAU_PARALLEL_SUPPORT:
            row += f"  {bp['plat_pow']:>+8.3f}"
        row += f"  {dt:>8}"
        print(row)


print_bm_table(bm_major, "MAJOR")
print_bm_table(bm_minor, "MINOR")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 5: COMPOSITE MODEL + R²
# ══════════════════════════════════════════════════════════════════════════════
#
# The model price at any time t is:
#   log10(price) = log10(support(t))  +  Σ bubble_shape_i(t)
# which in linear price is:
#   price = support(t) × 10^[Σ bubble_shape_i(t)]
#
# R² is computed on the full daily time series in log10-price space.
# We compare two models: support-only (baseline) and support + all bubbles.
# ══════════════════════════════════════════════════════════════════════════════

# Dense plotting grid (years since genesis)
years_plot_bm       = np.linspace(PLOT_YEARS_MIN, PLOT_YEARS_MAX, PLOT_GRID_POINTS)
log_t_plot_bm       = np.log10(years_plot_bm)
log_support_plot_bm = intercept_sup + slope_sup * log_t_plot_bm   # log10(support)
support_plot_bm     = 10 ** log_support_plot_bm                    # support in USD


def bm_total_bubble(years, params_list):
    """
    Sum bubble_shape contributions (log-excess) from all bubbles in params_list.

    Because log10(price/support) is additive across independent bubble events,
    we simply sum each bubble's contribution at each point in time.
    Returns an array of shape (len(years),).
    """
    total = np.zeros(len(years))
    for bp in params_list:
        total += bubble_shape(years, bp['t_rise'], bp['r'],
                              bp['t_plateau'], bp['t_decay'], bp['d'],
                              bp.get('plat_pow', 0.0))
    return total


# ── Plotting-grid composites ─────────────────────────────────────────────────
bm_maj_plot   = bm_total_bubble(years_plot_bm, bm_major)
bm_min_plot   = bm_total_bubble(years_plot_bm, bm_minor)
bm_total_plot = bm_maj_plot + bm_min_plot
bm_composite  = 10 ** (log_support_plot_bm + bm_total_plot)   # USD price curve

# ── R² on the full daily time series ─────────────────────────────────────────
total_all_bm     = (bm_total_bubble(years_all, bm_major) +
                    bm_total_bubble(years_all, bm_minor))
composite_all_bm = log_support_all + total_all_bm
ss_tot           = np.sum((log_p_all - np.mean(log_p_all)) ** 2)
bm_r2_support    = 1 - np.sum((log_p_all - log_support_all)  ** 2) / ss_tot
bm_r2_comp       = 1 - np.sum((log_p_all - composite_all_bm) ** 2) / ss_tot

print(f"\n{'='*80}\nMODEL FIT QUALITY\n{'='*80}")
print(f"  R² support only:          {bm_r2_support:.6f}")
print(f"  R² support + all bubbles: {bm_r2_comp:.6f}   ΔR² = {bm_r2_comp - bm_r2_support:.6f}")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 6: PREDICT FUTURE BUBBLES
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*80}")
print("STEP 6: PREDICTING FUTURE BUBBLES")
print(f"{'='*80}")


def predict_future_bubbles(hist_params, n_future, extrap_weights,
                            use_interval_trend, label='', default_interval=3.8,
                            min_interval=0.15, mode='extrap', last_n=None):
    """Project future bubble parameters from historical bubbles.

    mode='avg'
        Every predicted bubble gets the weighted average of the last `last_n`
        historical bubbles.  All predictions are identical in shape; only the
        start time advances by the average interval.  Conservative / stable.

    mode='extrap'
        Parameters are linearly extrapolated along their historical trend
        (r and d log-linearly; K, dur_plateau, plat_pow linearly), so each
        successive prediction follows the fitted slope.

    last_n
        If set, only the most-recent N historical bubbles are used for the
        average or extrapolation.  None = use all.

    Weights layout per bubble (chronological):
        [r_rise, d_decay, K, interval, dur_plateau, plat_pow]
    K is fitted directly; dur_rise is derived as K/r to avoid blowup.
    """
    future = []
    if last_n is not None:
        hist_params = hist_params[-last_n:]
    n_hist = len(hist_params)
    if n_hist < 1 or n_future < 1:
        if n_future > 0:
            print(f"  [{label}] No historical bubbles — cannot predict.")
        return future

    n_params   = 6
    expected_w = n_hist * n_params
    use_w = (extrap_weights is not None and len(extrap_weights) == expected_w)
    if extrap_weights is not None and not use_w:
        print(f"  [{label}] Weight length mismatch: expected {expected_w}, "
              f"got {len(extrap_weights)}. Using uniform.")
    w = (np.array(extrap_weights, dtype=float).reshape(n_hist, n_params)
         if use_w else np.ones((n_hist, n_params)))

    starts        = [b['t_rise'] for b in hist_params]
    intervals_arr = np.array([starts[i+1] - starts[i] for i in range(len(starts) - 1)]
                             if n_hist >= 2 else [default_interval])
    intv_weights  = w[:len(intervals_arr), 3]

    vals_r  = np.array([b['r']           for b in hist_params])
    vals_d  = np.array([b['d']           for b in hist_params])
    vals_K  = np.array([b['K']           for b in hist_params])
    vals_dp = np.array([b['dur_plateau'] for b in hist_params])
    vals_pp = np.array([b.get('plat_pow', 0.0) for b in hist_params])

    def wextrap(vals, wi, target):
        x = np.arange(len(vals), dtype=float)
        c = np.polyfit(x, vals, 1, w=wi) if len(vals) >= 2 else [0.0, float(vals[0])]
        return float(np.polyval(c, target)), c

    def wavg(vals, wi):
        return float(np.average(vals, weights=wi))

    last_n_str = f'last {last_n}' if last_n is not None else 'all'
    print(f"\n  [{label}] mode={mode}  using {n_hist} bubbles ({last_n_str})")
    for lbl2, vals, wi, log_sc in [
            ('r',           vals_r,  w[:, 0], True),
            ('d',           vals_d,  w[:, 1], True),
            ('K',           vals_K,  w[:, 2], True),
            ('dur_plateau', vals_dp, w[:, 4], False),
            ('plat_pow',    vals_pp, w[:, 5], False)]:
        fit_v = np.log(np.maximum(vals, 1e-9)) if log_sc else vals
        data_str = ['%.3f' % v for v in vals]
        if mode == 'extrap':
            _, c = wextrap(fit_v, wi, 0)
            suffix = ' [log-lin]' if log_sc else ''
            print(f"    {lbl2:12s}: slope={c[0]:+.4f}  data={data_str}{suffix}")
        else:
            avg_v = wavg(fit_v, wi)
            disp  = np.exp(avg_v) if log_sc else avg_v
            print(f"    {lbl2:12s}: avg={disp:.4f}  data={data_str}")
    if use_interval_trend and len(intervals_arr) >= 2:
        _, c = wextrap(intervals_arr, intv_weights, 0)
        print(f"    {'interval':12s}: slope={c[0]:+.4f}  "
              f"data={['%.3f'%v for v in intervals_arr]}  [trend]")
    else:
        wt_avg = float(np.average(intervals_arr, weights=intv_weights))
        print(f"    {'interval':12s}: weighted avg={wt_avg:.3f}  "
              f"data={['%.3f'%v for v in intervals_arr]}  [avg]")

    # Pre-compute avg-mode fixed parameters (same for every prediction)
    if mode == 'avg':
        fixed_r  = np.exp(wavg(np.log(np.maximum(vals_r, 1e-9)), w[:, 0]))
        fixed_d  = np.exp(wavg(np.log(np.maximum(vals_d, 1e-9)), w[:, 1]))
        fixed_K  = np.exp(wavg(np.log(np.maximum(vals_K, 1e-9)), w[:, 2]))
        fixed_dp = wavg(vals_dp, w[:, 4])
        fixed_pp = float(np.clip(wavg(vals_pp, w[:, 5]), -PLAT_POW_RANGE, PLAT_POW_RANGE))

    last_start = starts[-1]
    for j in range(n_future):
        tgt = n_hist + j
        if mode == 'avg':
            pred_r, pred_d, pred_K, pred_dp, pred_pp = \
                fixed_r, fixed_d, fixed_K, fixed_dp, fixed_pp
        else:
            _lr, _ = wextrap(np.log(np.maximum(vals_r, 1e-9)), w[:, 0], tgt)
            pred_r = np.exp(_lr)
            _ld, _ = wextrap(np.log(np.maximum(vals_d, 1e-9)), w[:, 1], tgt)
            pred_d = np.exp(_ld)
            _lk, _ = wextrap(np.log(np.maximum(vals_K, 1e-9)), w[:, 2], tgt)
            pred_K = np.exp(_lk)
            pred_dp, _ = wextrap(vals_dp, w[:, 4], tgt)
            pred_pp, _ = wextrap(vals_pp, w[:, 5], tgt)
            pred_pp    = float(np.clip(pred_pp, -PLAT_POW_RANGE, PLAT_POW_RANGE))

        if use_interval_trend and len(intervals_arr) >= 2:
            pred_intv, _ = wextrap(intervals_arr, intv_weights, len(intervals_arr) + j)
        else:
            pred_intv = float(np.average(intervals_arr, weights=intv_weights))

        pred_K    = max(pred_K,    0.01)
        pred_dp   = max(pred_dp,   0.0)
        pred_intv = max(pred_intv, min_interval)

        pred_tr = (last_start if j == 0 else future[-1]['t_rise']) + pred_intv
        if j == 0:
            while pred_tr <= years_all[-1]:
                pred_tr += pred_intv
        pred_dr  = max(pred_K / pred_r, 0.02)
        pred_tpl = pred_tr + pred_dr
        pred_tdc = pred_tpl + pred_dp
        pred_K_end  = (pred_K + pred_pp * np.log10(max(pred_tdc / pred_tpl, 1e-12))
                       if pred_dp > 0 else pred_K)
        peak_K_disp = max(pred_K, pred_K_end)

        sup_at_pk = A_sup * pred_tpl ** B_sup
        peak_px   = sup_at_pk * 10 ** peak_K_disp

        fb = {
            't_rise': pred_tr, 't_start': pred_tr,
            'r': pred_r, 't_plateau': pred_tpl,
            't_decay': pred_tdc, 'd': pred_d,
            'K': peak_K_disp, 'K_peak': pred_K,
            'K_end': pred_K_end, 'plat_pow': pred_pp,
            'dur_rise': pred_dr, 'dur_plateau': pred_dp,
            'interval': pred_intv,
            'date_start': genesis + pd.Timedelta(days=pred_tr  * 365.25),
            'date_plat':  genesis + pd.Timedelta(days=pred_tpl * 365.25),
            'date_decay': genesis + pd.Timedelta(days=pred_tdc * 365.25),
        }
        future.append(fb)
        intv_mode = 'trend' if use_interval_trend else 'avg'
        print(f"    Predicted {label} #{n_hist+j+1}:  "
              f"t_rise={pred_tr:.2f} yr  ({fb['date_start'].strftime('%Y-%m-%d')})  "
              f"interval={pred_intv:.2f} yr [{intv_mode}]  "
              f"K={peak_K_disp:.3f} ({10**peak_K_disp:.1f}×)  "
              f"plat_pow={pred_pp:+.2f}  peak≈${peak_px:,.0f}")
    return future


bm_future_major = predict_future_bubbles(
    bm_major, N_PREDICT_MAJOR,
    MAJOR_EXTRAP_WEIGHTS, MAJOR_INTERVAL_USE_TREND,
    label='MAJOR', default_interval=MAJOR_INTERVAL_YR, min_interval=1.4,
    mode=PREDICT_MODE, last_n=PREDICT_LAST_N_MAJOR)

bm_future_minor = []
if N_PREDICT_MINOR > 0:
    bm_future_minor = predict_future_bubbles(
        bm_minor, N_PREDICT_MINOR,
        MINOR_EXTRAP_WEIGHTS, MINOR_INTERVAL_USE_TREND,
        label='MINOR', default_interval=MAJOR_INTERVAL_YR, min_interval=0.15,
        mode=PREDICT_MODE, last_n=PREDICT_LAST_N_MINOR)

# Composite including predicted bubbles (for plotting)
bm_future_total = bm_total_plot.copy()
for fb in bm_future_major + bm_future_minor:
    bm_future_total += bubble_shape(years_plot_bm, fb['t_rise'], fb['r'],
                                    fb['t_plateau'], fb['t_decay'], fb['d'],
                                    fb.get('plat_pow', 0.0))
bm_composite_future = 10 ** (log_support_plot_bm + bm_future_total)

print(f"\n{'='*80}")


# ══════════════════════════════════════════════════════════════════════════════
# PLOT HELPERS
# ══════════════════════════════════════════════════════════════════════════════
_exp_bm       = np.arange(-2, 10)
_maj_ticks_bm = 10.0 ** _exp_bm
_min_ticks_bm = [s * 10.0 ** e for e in _exp_bm for s in range(2, 10)]
_every_yr_bm  = np.arange(1, 43)
_xtv_bm       = [1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35, 40]
_xtl_bm       = [f"{y}\n{2009+y}" for y in _xtv_bm]
_xtv_lin_bm   = list(range(1, 18, 2))
_xtl_lin_bm   = [f"{y}\n({2009+y})" for y in _xtv_lin_bm]


def setup_loglog_bm(ax):
    """
    Configure ax for a log-log Bitcoin price chart.

    Both axes are logarithmic.  Y-axis shows USD price with dollar formatting;
    X-axis shows years since genesis with labels at round years.  Minor gridlines
    mark every integer year so individual halving cycles are visible.
    """
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.yaxis.set_major_locator(FixedLocator(_maj_ticks_bm))
    ax.yaxis.set_major_formatter(StrMethodFormatter('${x:,.0f}'))
    ax.yaxis.set_minor_locator(FixedLocator(_min_ticks_bm))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xticks(_xtv_bm); ax.set_xticklabels(_xtl_bm, fontsize=8)
    ax.xaxis.set_minor_locator(FixedLocator(_every_yr_bm))
    ax.grid(which='major', color=GRID_MAJOR_COLOR, lw=0.6, alpha=0.6)
    ax.grid(which='minor', color=GRID_MINOR_COLOR, lw=0.3, alpha=0.4)
    ax.set_facecolor(PLOT_BG_COLOR)


def save_plot_bm(basename):
    """Save the current figure as both vector (SVG) and raster (JPG at 200 dpi)."""
    plt.savefig(f'{basename}.svg', format='svg', bbox_inches='tight')
    plt.savefig(f'{basename}.jpg', format='jpg', bbox_inches='tight', dpi=200)
    print(f"  Saved {basename}.svg/.jpg")


def set_linear_xticks_bm(ax):
    """Apply the standard linear-axis year labels to ax."""
    ax.set_xticks(_xtv_lin_bm)
    ax.set_xticklabels(_xtl_lin_bm, fontsize=8)


def draw_bubbles_bm(ax, params, colors, ls='-', lw=1.2, alpha=0.65, label_prefix=''):
    """Draw individual bubble price curves on ax using the dense plotting grid."""
    if not params or not colors:
        return
    n_c = len(colors)
    for i, bp in enumerate(params):
        bub = bubble_shape(years_plot_bm, bp['t_rise'], bp['r'],
                           bp['t_plateau'], bp['t_decay'], bp['d'],
                           bp.get('plat_pow', 0.0))
        mask = bub > 0.001
        if mask.any():
            ax.plot(years_plot_bm[mask], 10 ** (log_support_plot_bm[mask] + bub[mask]),
                    color=colors[i % n_c], ls=ls, lw=lw, alpha=alpha,
                    label=f"{label_prefix}{i+1}" if i < 8 else None)


# ══════════════════════════════════════════════════════════════════════════════
# PLOT A: LOG-LOG PRICE CHART WITH COMPOSITE MODEL
# Both axes are logarithmic.  On a log-log chart a pure power law (support line)
# appears as a straight line.  Each bubble's individual price trajectory is
# plotted (major = solid, minor = dotted); the thick red composite is their sum
# translated back to USD: price = support(t) × 10^[Σ bubble_shape_i(t)].
# Predicted bubbles are shown with dashed lines.
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=FIGSIZE_WIDE)
setup_loglog_bm(ax)

ax.scatter(years_all, y_all, color=DATA_COLOR, alpha=0.4, s=1.5,
           edgecolor='none', label='Historical data', zorder=3)
ax.plot(years_plot_bm, support_plot_bm, color=SUPPORT_COLOR, ls='--', lw=1.5, zorder=5,
        label=f'Support (B={B_sup:.3f})')

draw_bubbles_bm(ax, bm_major, MAJOR_COLORS, ls='-',  lw=1.3, alpha=0.75, label_prefix='Major ')
draw_bubbles_bm(ax, bm_minor, MINOR_COLORS, ls=':',  lw=1.0, alpha=0.65, label_prefix='Minor ')
draw_bubbles_bm(ax, bm_future_major, MAJOR_COLORS[len(bm_major):],
                ls='--', lw=1.5, alpha=0.80, label_prefix='Pred Major ')
draw_bubbles_bm(ax, bm_future_minor, MINOR_COLORS[len(bm_minor):],
                ls='-.', lw=1.2, alpha=0.75, label_prefix='Pred Minor ')

ax.plot(years_plot_bm, bm_composite_future, color=COMPOSITE_COLOR, ls='-', lw=1.8, zorder=8,
        label=f'Composite + predictions  R²={bm_r2_comp:.4f}')
ax.axvline(today_years, color=TODAY_COLOR, ls='-', lw=1.0, alpha=0.5, label='Today')
ax.set_ylim(PRICE_YMIN, PRICE_YMAX); ax.set_xlim(PLOT_YEARS_MIN, PLOT_YEARS_MAX)
ax.set_title(f'Manual Bubble Model — years {BUBBLE_YEARS}  '
             f'{len(bm_major)} Major + {len(bm_minor)} Minor  '
             f'(+{N_PREDICT_MAJOR} major predicted)')
ax.set_xlabel('Years since Bitcoin genesis (Jan 2009)')
ax.set_ylabel('Bitcoin Price (USD)')
ax.legend(loc='upper left', fontsize=8, framealpha=0.95, ncol=2,
          edgecolor=GRID_MAJOR_COLOR, facecolor=PLOT_BG_COLOR)
plt.tight_layout(pad=1.5)
save_plot_bm('bm_loglog')
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# PLOT B: LOG-LOG ZOOMED  (recent history + near-future prediction window)
# Same data as Plot A but restricted to ZOOM_XMIN…ZOOM_XMAX years so the
# predicted bubbles fill most of the frame.
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=FIGSIZE_WIDE)
setup_loglog_bm(ax)

ax.scatter(years_all, y_all, color=DATA_COLOR, alpha=0.55, s=5,
           edgecolor='none', label='Historical data', zorder=3)
ax.plot(years_plot_bm, support_plot_bm, color=SUPPORT_COLOR, ls='--', lw=1.5,
        label=f'Support (B={B_sup:.3f})')
draw_bubbles_bm(ax, bm_major, MAJOR_COLORS, ls='-',  lw=1.3, alpha=0.75, label_prefix='Major ')
draw_bubbles_bm(ax, bm_minor, MINOR_COLORS, ls=':',  lw=1.0, alpha=0.60, label_prefix='Minor ')
draw_bubbles_bm(ax, bm_future_major, MAJOR_COLORS[len(bm_major):],
                ls='--', lw=1.5, alpha=0.80, label_prefix='Pred Major ')
draw_bubbles_bm(ax, bm_future_minor, MINOR_COLORS[len(bm_minor):],
                ls='-.', lw=1.2, alpha=0.75, label_prefix='Pred Minor ')
ax.plot(years_plot_bm, bm_composite_future, color=COMPOSITE_COLOR, ls='-', lw=1.8,
        zorder=8, label=f'Composite (R²={bm_r2_comp:.4f})')
ax.axvline(today_years, color=TODAY_COLOR, ls='-', lw=1.0, alpha=0.5, label='Today')

ax.set_ylim(ZOOM_YMIN, ZOOM_YMAX); ax.set_xlim(ZOOM_XMIN, ZOOM_XMAX)
ax.set_title('Manual Bubble Model — Zoomed Projection')
ax.set_xlabel('Years since Bitcoin genesis (Jan 2009)')
ax.set_ylabel('Bitcoin Price (USD)')
ax.legend(loc='upper left', fontsize=8.5, framealpha=0.95, ncol=2,
          edgecolor=GRID_MAJOR_COLOR, facecolor=PLOT_BG_COLOR)
plt.tight_layout(pad=1.5)
save_plot_bm('bm_loglog_zoomed')
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# PLOT C: SEMI-LOG  (linear time axis, log price axis)
# The power-law support line becomes a curve here (not a straight line),
# which makes it easy to see how price acceleration slows as Bitcoin matures.
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=FIGSIZE_WIDE)
ax.set_facecolor(PLOT_BG_COLOR); ax.set_yscale('log')

ax.scatter(years_all, y_all, color=DATA_COLOR, alpha=0.4, s=1.5,
           edgecolor='none', label='Historical data', zorder=3)
ax.plot(years_plot_bm, support_plot_bm, color=SUPPORT_COLOR, ls='--', lw=1.5, label='Support')
ax.plot(years_plot_bm, bm_composite_future, color=COMPOSITE_COLOR, ls='-', lw=1.8,
        zorder=8, label='Composite + predictions')
draw_bubbles_bm(ax, bm_major, MAJOR_COLORS, ls='-',  lw=1.0, alpha=0.60, label_prefix='Major ')
draw_bubbles_bm(ax, bm_minor, MINOR_COLORS, ls=':',  lw=1.0, alpha=0.55, label_prefix='Minor ')
draw_bubbles_bm(ax, bm_future_major, MAJOR_COLORS[len(bm_major):],
                ls='--', lw=1.5, alpha=0.80, label_prefix='Pred Major ')
draw_bubbles_bm(ax, bm_future_minor, MINOR_COLORS[len(bm_minor):],
                ls='-.', lw=1.2, alpha=0.75, label_prefix='Pred Minor ')
ax.axvline(today_years, color=TODAY_COLOR, ls='-', lw=1.0, alpha=0.5, label='Today')

ax.yaxis.set_major_locator(FixedLocator(_maj_ticks_bm))
ax.yaxis.set_major_formatter(StrMethodFormatter('${x:,.0f}'))
ax.yaxis.set_minor_locator(FixedLocator(_min_ticks_bm))
x_ticks_lin = list(range(1, 43, 2))
ax.set_xticks(x_ticks_lin)
ax.set_xticklabels([f"{y}\n({2009+y})" for y in x_ticks_lin], fontsize=9)
ax.set_xlim(PLOT_YEARS_MIN, PLOT_YEARS_MAX); ax.set_ylim(PRICE_YMIN, PRICE_YMAX)
ax.grid(which='major', color=GRID_MAJOR_COLOR, lw=0.6, alpha=0.6)
ax.grid(which='minor', axis='y', color=GRID_MINOR_COLOR, lw=0.3, alpha=0.4)
ax.set_xlabel('Years since Bitcoin genesis (Jan 2009)')
ax.set_ylabel('Bitcoin Price (USD) — log scale')
ax.set_title('Manual Bubble Model — Semi-Log View')
ax.legend(loc='upper left', fontsize=9, framealpha=0.95,
          edgecolor=GRID_MAJOR_COLOR, facecolor=PLOT_BG_COLOR)
plt.tight_layout(pad=1.5)
save_plot_bm('bm_semilog')
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# PLOT D: BUBBLE DECOMPOSITION IN LOG-EXCESS SPACE
# Shows individual bubble components vs the smoothed actual log-excess.
#
# Y-axis is log10(price/support) — the "excess" above the support power law.
# Each coloured curve is one bubble's bubble_shape() contribution.  Their sum
# is the composite model's total log-excess.
#
# The dotted black line is a 20-day rolling mean of the actual log-excess; it
# serves as the "ground truth" each bubble is trying to explain.  Because
# bubbles are fitted sequentially to residuals, they should not overlap
# vertically — the dominant bubble captures most of the shared signal first.
# ══════════════════════════════════════════════════════════════════════════════
smooth_all_bm = (pd.Series(log_excess_all)
                 .rolling(20, center=True, min_periods=10).mean().values)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=FIGSIZE_DECOMP, sharex=True)

# ── Major bubbles panel ──────────────────────────────────────────────────────
ax1.axhline(0, color=REFERENCE_LINE_COLOR, lw=0.8, alpha=0.5)
for i, bp in enumerate(bm_major):
    bub = bubble_shape(years_plot_bm, bp['t_rise'], bp['r'],
                       bp['t_plateau'], bp['t_decay'], bp['d'], bp.get('plat_pow', 0.0))
    c   = MAJOR_COLORS[i % len(MAJOR_COLORS)]
    ax1.fill_between(years_plot_bm, 0, bub, color=c, alpha=0.25)
    ax1.plot(years_plot_bm, bub, color=c, lw=1.6,
             label=f'Major {i+1} ({bp.get("bubble_year","?")})'
                   f'  K={bp["K"]:.2f} ({10**bp["K"]:.1f}×)')
ax1.plot(years_all, np.maximum(0, smooth_all_bm), color='black', lw=0.8,
         alpha=0.4, ls=':', label='Actual (20-d smooth)')
ax1.set_ylabel('log₁₀ excess'); ax1.set_title('Manual Model — MAJOR Bubble Components')
ax1.legend(loc='upper left', fontsize=8, ncol=2); ax1.grid(True, alpha=0.3)
ax1.axvline(today_years, color=TODAY_COLOR, ls='-', lw=1.0, alpha=0.5)

# ── Minor bubbles panel ──────────────────────────────────────────────────────
ax2.axhline(0, color=REFERENCE_LINE_COLOR, lw=0.8, alpha=0.5)
for i, bp in enumerate(bm_minor):
    bub = bubble_shape(years_plot_bm, bp['t_rise'], bp['r'],
                       bp['t_plateau'], bp['t_decay'], bp['d'], bp.get('plat_pow', 0.0))
    c   = MINOR_COLORS[i % len(MINOR_COLORS)]
    ax2.fill_between(years_plot_bm, 0, bub, color=c, alpha=0.35)
    ax2.plot(years_plot_bm, bub, color=c, lw=1.5,
             label=f'Minor {i+1} ({bp.get("bubble_year","?")})'
                   f'  K={bp["K"]:.2f} ({10**bp["K"]:.1f}×)')
if not bm_minor:
    ax2.text(0.5, 0.5, 'No minor bubbles detected',
             transform=ax2.transAxes, ha='center', va='center',
             fontsize=12, color=FALLBACK_TEXT_COLOR)
ax2.plot(years_all, np.maximum(0, smooth_all_bm), color='black', lw=0.8,
         alpha=0.4, ls=':', label='Actual (20-d smooth)')
ax2.set_ylabel('log₁₀ excess'); ax2.set_title('Manual Model — MINOR Bubble Components')
ax2.legend(loc='upper left', fontsize=8, ncol=2); ax2.grid(True, alpha=0.3)
ax2.axvline(today_years, color=TODAY_COLOR, ls='-', lw=1.0, alpha=0.5)
ax2.set_xlabel('Years since Bitcoin genesis (Jan 2009)')

for ax in (ax1, ax2):
    set_linear_xticks_bm(ax)
plt.tight_layout()
save_plot_bm('bm_decomposition')
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# PLOT E: RESIDUALS
# Top panel:    log-excess residual from support-only model
#               = log10(price) − log10(support)  = log10(price/support)
#               This is just the raw log-excess; a perfect model would show
#               zero residual throughout.  Positive spikes are bubbles.
#
# Bottom panel: residual after subtracting the full manual-bubble composite
#               = log10(price) − [log10(support) + Σ bubble_shape_i(t)]
#               Ideally this is white noise centred on zero.  Systematic
#               structure indicates events the model did not capture.
#
# The 90-day rolling mean in both panels highlights slow trends the model misses.
# ══════════════════════════════════════════════════════════════════════════════
bm_resid = log_p_all - composite_all_bm

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=FIGSIZE_RESIDUALS, sharex=True)

ax1.scatter(years_all, log_excess_all, color=SCATTER_COLOR, alpha=0.2, s=1)
s_sm_bm = pd.Series(log_excess_all).rolling(RESIDUAL_SMOOTH_WINDOW, center=True, min_periods=30).mean().values
ax1.plot(years_all, s_sm_bm, color=RESID_SUP_COLOR, lw=1.5)
ax1.axhline(0, color=REFERENCE_LINE_COLOR, lw=1, alpha=0.5)
ax1.set_ylabel('Residual (log₁₀)')
ax1.set_title(f'Support-Only Residuals  (R²={bm_r2_support:.4f})')
ax1.grid(True, alpha=0.3)

ax2.scatter(years_all, bm_resid, color=SCATTER_COLOR, alpha=0.2, s=1)
c_sm_bm = pd.Series(bm_resid).rolling(RESIDUAL_SMOOTH_WINDOW, center=True, min_periods=30).mean().values
ax2.plot(years_all, c_sm_bm, color=RESID_COMP_COLOR, lw=1.5)
ax2.axhline(0, color=REFERENCE_LINE_COLOR, lw=1, alpha=0.5)
ax2.set_xlabel('Years since Bitcoin genesis (Jan 2009)')
ax2.set_ylabel('Residual (log₁₀)')
ax2.set_title(f'Manual Model Residuals  (R²={bm_r2_comp:.4f})')
ax2.grid(True, alpha=0.3)

max_r_bm = max(np.nanmax(np.abs(s_sm_bm[~np.isnan(s_sm_bm)])),
               np.nanmax(np.abs(c_sm_bm[~np.isnan(c_sm_bm)]))) * 1.2
ax1.set_ylim(-max_r_bm, max_r_bm); ax2.set_ylim(-max_r_bm, max_r_bm)
for ax in (ax1, ax2):
    set_linear_xticks_bm(ax)
plt.tight_layout()
save_plot_bm('bm_residuals')
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# PLOT F: PEAK-ALIGNED ENVELOPE — MAJOR BUBBLES + PREDICTED MAJORS
# Each bubble is replotted with time centred on t_plateau (the peak moment).
# Aligning peaks makes the rise and decay shapes directly comparable across
# all cycles regardless of when they occurred in calendar time.
#
# Y-axis: log₁₀(price / support) — the same bubble_shape() value used
# throughout the model.  The peak of each curve sits at its K value, so the
# declining K trend is immediately visible across cycles.
#
# Grey envelope: min/max band across all historical major bubbles at each
# time offset.  A tight band means consistent shape; a wide band means the
# model shape varied significantly across cycles.
#
# The vertical dashed line at t=0 marks t_plateau (where the rise ends and
# the plateau/decay begins).  Predicted bubbles are drawn dashed.
# ══════════════════════════════════════════════════════════════════════════════
t_rel_env = np.linspace(-4, 8, 3000)   # years relative to t_plateau

# Collect all curve arrays and metadata for both historical and predicted majors
env_curves  = []   # log-excess arrays on t_rel_env
env_colors  = []
env_labels  = []
env_is_pred = []

for i, bp in enumerate(bm_major):
    t_abs = t_rel_env + bp['t_plateau']
    bub = bubble_shape(t_abs, bp['t_rise'], bp['r'],
                       bp['t_plateau'], bp['t_decay'], bp['d'],
                       bp.get('plat_pow', 0.0))
    env_curves.append(bub)
    env_colors.append(MAJOR_COLORS[i % len(MAJOR_COLORS)])
    env_labels.append(f"Major {i+1} ({bp.get('bubble_year', '?')})  K={bp['K']:.3f}")
    env_is_pred.append(False)

for i, fb in enumerate(bm_future_major):
    t_abs = t_rel_env + fb['t_plateau']
    bub = bubble_shape(t_abs, fb['t_rise'], fb['r'],
                       fb['t_plateau'], fb['t_decay'], fb['d'],
                       fb.get('plat_pow', 0.0))
    env_curves.append(bub)
    env_colors.append(MAJOR_COLORS[(len(bm_major) + i) % len(MAJOR_COLORS)])
    env_labels.append(f"Pred Major {len(bm_major)+i+1}  K={fb['K']:.3f}")
    env_is_pred.append(True)

fig, ax = plt.subplots(figsize=FIGSIZE_ENVELOPE)
ax.set_facecolor(PLOT_BG_COLOR)

# Grey envelope spanning the range of historical major bubbles only
n_hist_maj = len(bm_major)
if n_hist_maj > 1:
    hist_arr = np.array(env_curves[:n_hist_maj])
    env_lo   = hist_arr.min(axis=0)
    env_hi   = hist_arr.max(axis=0)
    # Only shade where at least one historical bubble is non-zero
    nonzero = env_hi > 0.001
    ax.fill_between(t_rel_env, env_lo, env_hi, where=nonzero,
                    color='#DDDDDD', alpha=0.55, zorder=1, label='Historical range')

# Draw each bubble curve
for curve, color, label, is_pred in zip(env_curves, env_colors, env_labels, env_is_pred):
    mask = curve > 0.001
    if not mask.any():
        continue
    ax.plot(t_rel_env[mask], curve[mask],
            color=color,
            ls='--' if is_pred else '-',
            lw=1.8 if is_pred else 1.5,
            alpha=0.9,
            label=label,
            zorder=3 if is_pred else 2)

# Reference lines
ax.axvline(0, color=REFERENCE_LINE_COLOR, ls='--', lw=1.0, alpha=0.6,
           label='Peak (t = t_plateau)', zorder=4)
ax.axhline(0, color=REFERENCE_LINE_COLOR, ls='-', lw=0.7, alpha=0.35, zorder=1)

ax.set_xlabel('Years relative to bubble peak (t − t_plateau)')
ax.set_ylabel('log₁₀(price / support)')
ax.set_title('Peak-Aligned Major Bubble Shapes  '
             '(dashed = predicted; grey band = historical range)')
ax.set_xlim(t_rel_env[0], t_rel_env[-1])
ax.set_ylim(-0.05, None)
ax.legend(loc='upper right', fontsize=9, framealpha=0.95,
          edgecolor=GRID_MAJOR_COLOR, facecolor=PLOT_BG_COLOR)
ax.grid(True, color=GRID_MAJOR_COLOR, lw=0.5, alpha=0.5)
plt.tight_layout()
save_plot_bm('bm_envelope')
plt.show()


# ══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*80}")
print("FINAL SUMMARY — Manual Bubble Model")
print(f"{'='*80}")
print(f"\n  SUPPORT:  price = {A_sup:.4e} × t^{B_sup:.4f}")
print(f"\n  BUBBLE YEARS: {BUBBLE_YEARS}")
print(f"  Search window: ±{BUBBLE_YEAR_WINDOW} yr around Jan 1 of each year")
print(f"  Peaks found:   {len(bm_peaks)} of {len(BUBBLE_YEARS)} years had data in window")
print(f"\n  BUBBLE MODEL: bubble_shape() — exponential rise/fall, power-law plateau")
print(f"  Plateau      : {'parallel to support (plat_pow = 0)' if PLATEAU_PARALLEL_SUPPORT else 'free power-law tilt (plat_pow optimised)'}")
print(f"\n  FITTED:    {len(bm_major)} MAJOR + {len(bm_minor)} MINOR = {n_det} total")
print(f"  PREDICTED: {N_PREDICT_MAJOR} MAJOR + {N_PREDICT_MINOR} MINOR future bubbles")
if bm_future_major:
    for j, fb in enumerate(bm_future_major):
        print(f"    Pred Major #{len(bm_major)+j+1}:  "
              f"{fb['date_start'].strftime('%Y-%m-%d')}  K={fb['K']:.3f}  "
              f"peak≈${A_sup * fb['t_plateau']**B_sup * 10**fb['K']:,.0f}")
print(f"\n  R² support only:          {bm_r2_support:.6f}")
print(f"  R² support + all bubbles: {bm_r2_comp:.6f}   ΔR² = {bm_r2_comp - bm_r2_support:.6f}")
print("=" * 80)


CSV loaded.  Shape: (5686, 2)
Date range  : 2010-07-17 → 2026-02-08
Fit window  : 5686 points
Price range : $0.05 – $124,641.52
Today       : t = 17.123 yr

STEP 1: FITTING SUPPORT LINE (bottom 25% OLS residual)
  Support points : 1422 / 5686
  Power law      : price = 4.9132e-03 × t^5.7842

STEP 2: LOCATE BUBBLE PEAKS  (BUBBLE_YEAR_WINDOW = ±0.5 yr)

  Searching for peaks near: [2013, 2017, 2021, 2025]

  Year  t_center           Search window     Peak date   t_peak   Raw K
  ------------------------------------------------------------------------
  2013     3.978  [   3.478,    4.478]    2013-04-10    4.249   1.024
  2017     7.978  [   7.478,    8.478]    2017-06-11    8.419   0.419
  2021    11.978  [  11.478,   12.478]    2021-04-16   12.266   0.854
  2025    15.978  [  15.478,   16.478]    2024-12-17   15.937   0.381

STEP 3: FITTING BUBBLE SHAPES
  plat_pow: free parameter
  FIT_CONTEXT_YR = 0.5  FIT_RISE_LOOKBACK_YR = 0.5
  DE_MAXITER = 2000  DE_POPSIZE = 18
  Fitting bubble ye